# Chat conversation


Let's see how we can create chat conversation between human and llm with some useful feature.

## Ollama Setup
So we will be using open source models from ollama as an llm/slm call to understand the concept

In [3]:
%%capture
# 1. Update system packages
! sudo apt update
# 2. Install pciutils (required for proper GPU detection)
! sudo apt install -y pciutils

In [4]:

# let's download the ollama executables
!curl -fsSL https://ollama.com/install.sh | sh


>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
# serve the ollama
!nohup ollama serve &

nohup: appending output to 'nohup.out'


In [7]:
# pull the llm/slm models
# !ollama pull smollm:135m
# !ollama pull embeddinggemma:300m
# !ollama pull tinyllama:1.1b
!ollama pull gemma3:270m
# !ollama pull gemma3:1b # good at natural language understading


In [6]:
# list the pulled models
!ollama list

NAME    ID    SIZE    MODIFIED 


In [8]:
!ollama ps

NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL 


## LangChain/LangGraph Framework
we will be using langchain framework to implement the agents

In [9]:
# Installing required libraries
%%capture

!pip install --upgrade langchain
!pip install --upgrade langchain-ollama
!pip install --upgrade langchain-core
!pip install --upgrade langchain-community
!pip install --upgrade langgraph
!pip install pprintpp

In [102]:

# importing required libs
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from pprint import pprint

In [11]:
# initializing model - using small lm whcih should satisfy the need here
gemma_llm = ChatOllama(model="gemma3:270m", temperature=0.5) # good at understanding in natural language processing

### Simple Chat Conversation without Chat Memory

In [12]:
# defining system prompt for the model

system_prompt = """Respond the the user query as a chat converstation manner."""

# defining the llm model as chain
chatModelChain = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{user_query}")
]) | gemma_llm


In [30]:
# let's chat now
while True:
  user_input = input("type to ask...\n")
  # specifying some keywords as an interruption keywords
  if user_input in ("quit", "bye", "end", "thanks", "thank you"):
    print("System:: Happy to help! Bye!!!")
    break

  print("AI:: ", chatModelChain.invoke({"user_query": user_input}).content )


type to ask...
Biggest planet in our solar system?
AI::  The biggest planet in our solar system is **Jupiter**.

type to ask...
what is the mass of it ?
AI::  Okay, I'm ready. What is the mass of it?

type to ask...
quit
System:: Happy to help! Bye!!!


### Chat Conversation with InMemory Cache


So we can give memory to out chat conversation such that model will get the context.

For example:

type to ask...

Biggest planet in our solar system?

AI::  The biggest planet in our solar system is **Jupiter**.

****************************************************
type to ask...

what is the mass of it ?

AI::  Okay, I'm ready. What is the mass of it?

****************************************************
type to ask...

quit

System:: Happy to help! Bye!!!
****************************************************



Here model did not get that I was reffering to "Jupiter"

In [14]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [22]:
# we need to define the cache mehanism

# dict to store the history for each session id/interaction id
store = {}

def get_session_history_from_dict(session_id: str) -> InMemoryChatMessageHistory:
    """Retrieves or creates a history object for a given session ID."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [23]:
# creating object to store memory history
inMemoryHistory = InMemoryChatMessageHistory()
# defining the llm model as chain to use memory
chatModelWithMemChain =  ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_query}")
])  | gemma_llm

chatModelWithMemRunnable = RunnableWithMessageHistory(
    runnable=chatModelWithMemChain,
    get_session_history=get_session_history_from_dict,
    history_messages_key="chat_history",
    input_messages_key="user_query"
)



In [17]:
interacted_sessions = []

In [31]:
# let's create the config to identify various sessions
import uuid
sid = str(uuid.uuid4())
interacted_sessions.append(sid)
print(interacted_sessions)


['0c49d102-085f-410a-84e2-ce5ea43556ef', '6dbf5f16-7666-4f2c-8d0f-3c683c7fb897', '6e05f6cd-8119-477b-8db8-4179032f711c']


In [32]:
config = {"configurable":{"session_id":sid}}

print("Running for this session Id:", sid)
# let's chat now
while True:
  user_input = input("type to ask...\n")
  # specifying some keywords as an interruption keywords
  if user_input in ("quit", "bye", "end", "thanks", "thank you"):
    print("System:: Happy to help! Bye!!!")
    break

  # adding additional config to help to get the session id
  print("AI:: ", chatModelWithMemRunnable.invoke(input={"user_query": user_input}, config=config ).content)


Running for this session Id: 6e05f6cd-8119-477b-8db8-4179032f711c
type to ask...
Biggest planet in our solar system?
AI::  That's a great question! The biggest planet in our solar system is **Jupiter**.

type to ask...
what is the mass of it ?
AI::  The mass of Jupiter is approximately 35 times the mass of the Earth.
type to ask...
quit
System:: Happy to help! Bye!!!


Nice:

here we saw, it is able to give the answer for follow-up question without loosing the context.


Running for this session Id: 6e05f6cd-8119-477b-8db8-4179032f711c
type to ask...

Biggest planet in our solar system?

AI::  That's a great question! The biggest planet in our solar system is **Jupiter**.

******************************************
type to ask...

what is the mass of it ?

AI::  The mass of Jupiter is approximately 35 times the mass of the Earth.
******************************************

type to ask...

quit

System:: Happy to help! Bye!!!

Let's analyze our store dict

In [36]:
store[interacted_sessions[-1]]

InMemoryChatMessageHistory(messages=[HumanMessage(content='Biggest planet in our solar system?', additional_kwargs={}, response_metadata={}), AIMessage(content="That's a great question! The biggest planet in our solar system is **Jupiter**.\n", additional_kwargs={}, response_metadata={'model': 'gemma3:270m', 'created_at': '2025-11-18T11:16:43.266256293Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1198779522, 'load_duration': 226995625, 'prompt_eval_count': 30, 'prompt_eval_duration': 124070978, 'eval_count': 20, 'eval_duration': 778354069, 'logprobs': None, 'model_name': 'gemma3:270m', 'model_provider': 'ollama'}, id='lc_run--bf80b9e7-90ac-4f14-9e36-70b541ef498d-0', usage_metadata={'input_tokens': 30, 'output_tokens': 20, 'total_tokens': 50}), HumanMessage(content='what is the mass of it ?', additional_kwargs={}, response_metadata={}), AIMessage(content='The mass of Jupiter is approximately 35 times the mass of the Earth.', additional_kwargs={}, response_metadata={'model':

So here is the history of the current session.

**Note:** We can also store this into db or file system.

### Chat Conversation with Persiste File Memory


In [48]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import messages_from_dict, message_to_dict, BaseMessage
import os
import json
from typing import Sequence


In [88]:
# this is the default implementation of a class inherited from BaseChatMessageHistory
class FileChatMessageHistory(BaseChatMessageHistory):
    storage_path: str
    session_id: str

    @property
    def messages(self) -> list[BaseMessage]:
        try:
            with open(
                os.path.join(self.storage_path, self.session_id + ".json"),
                "r",
                encoding="utf-8",
            ) as f:
                messages_data = json.load(f)
            return messages_from_dict(messages_data)
        except FileNotFoundError:
            return []

    def add_messages(self, messages: Sequence[BaseMessage]) -> None:
        all_messages = list(self.messages)  # Existing messages
        all_messages.extend(messages)  # Add new messages

        serialized = [message_to_dict(message) for message in all_messages]
        file_path = os.path.join(self.storage_path, self.session_id + ".json")
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(serialized, f)

    def clear(self) -> None:
        file_path = os.path.join(self.storage_path, self.session_id + ".json")
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump([], f)

In [89]:
# a method to get the file from directory to get the history
def get_session_history_from_file(session_id:str):
  # print(f"inside get_session_history_from_file, input - ",session_id)
  fObj =  FileChatMessageHistory()
  fObj.session_id = session_id
  fObj.storage_path = f"{os.getcwd()}/user_interactions"

  return fObj



In [90]:

# creating object to store memory history
fileMemoryHistory = FileChatMessageHistory()
# defining the llm model as chain to use memory
chatModelWithPersistMemChain =  ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_query}")
])  | gemma_llm

chatModelWithPersistMemRunnable = RunnableWithMessageHistory(
    runnable=chatModelWithPersistMemChain,
    get_session_history=get_session_history_from_file,
    history_messages_key="chat_history",
    input_messages_key="user_query"
)



In [91]:
_all_interacted_sessions = []

In [92]:
# let's create the config to identify various sessions
import uuid
fsid = str(uuid.uuid4())
_all_interacted_sessions.append(fsid)
print(_all_interacted_sessions)


['6108fae5-7b40-4011-be76-130af36d813c']


In [93]:
config = {"configurable":{"session_id":fsid}}

print("Running for this file session Id:", fsid)
# let's chat now
while True:
  user_input = input("type to ask...\n")
  # specifying some keywords as an interruption keywords
  if user_input in ("quit", "bye", "end", "thanks", "thank you"):
    print("System:: Happy to help! Bye!!!")
    break

  # adding additional config to help to get the session id
  print("AI:: ", chatModelWithPersistMemRunnable.invoke(input={"user_query": user_input}, config=config ).content)


Running for this file session Id: 6108fae5-7b40-4011-be76-130af36d813c
type to ask...
What is the capital of Australia
AI::  The capital of Australia is Canberra.

type to ask...
Where is canberra?
AI::  Canberra is located in the Australian Capital Territory (ACT).
type to ask...
Where is Taj Mahal?
AI::  Taj Mahal is located in Agra, India.
type to ask...
quit
System:: Happy to help! Bye!!!


Nice, let's check is we persis the data into file or not

In [94]:
os.listdir()

['.config', 'nohup.out', 'user_interactions', 'sample_data']

In [95]:
os.listdir('user_interactions')

['11221d75-35aa-4c59-999f-9d2968fabb36',
 '6108fae5-7b40-4011-be76-130af36d813c.json']

In [103]:
with open(os.path.join(os.getcwd(), "user_interactions", _all_interacted_sessions[-1] + ".json"), "r") as jf:
  pprint(json.loads(jf.read()))

[{'data': {'additional_kwargs': {},
           'content': 'What is the capital of Australia',
           'id': None,
           'name': None,
           'response_metadata': {},
           'type': 'human'},
  'type': 'human'},
 {'data': {'additional_kwargs': {},
           'content': 'The capital of Australia is Canberra.\n',
           'id': 'lc_run--b0a52da5-92dc-447c-8e78-2c2a8dd26666-0',
           'invalid_tool_calls': [],
           'name': None,
           'response_metadata': {'created_at': '2025-11-18T12:02:40.804919488Z',
                                 'done': True,
                                 'done_reason': 'stop',
                                 'eval_count': 9,
                                 'eval_duration': 352497380,
                                 'load_duration': 226321556,
                                 'logprobs': None,
                                 'model': 'gemma3:270m',
                                 'model_name': 'gemma3:270m',
                 

Nice, we can use this for future interactions too.

### Testing

In [104]:
user1= "marvel_1"
config = {"configurable":{"session_id":user1}}

print("Running for this user:", user1)
# let's chat now
while True:
  user_input = input("type to ask...\n")
  # specifying some keywords as an interruption keywords
  if user_input in ("quit", "bye", "end", "thanks", "thank you"):
    print("System:: Happy to help! Bye!!!")
    break

  # adding additional config to help to get the session id
  print("AI:: ", chatModelWithPersistMemRunnable.invoke(input={"user_query": user_input}, config=config ).content)


Running for this file session Id: 6108fae5-7b40-4011-be76-130af36d813c
type to ask...
Hi My name is Captan America. I want to know the weather today
AI::  Hi Captan America!

type to ask...
what is the capital of America
AI::  The capital of America is Washington, D.C.
type to ask...
what is tea ?
AI::  Tea is a warm, sweet, and often bitter beverage made from the leaves of the tea plant, *Camellia radiata*.
type to ask...
thanks
System:: Happy to help! Bye!!!


let's create another use

In [105]:
user2= "marvel_2"
config = {"configurable":{"session_id":user2}}

print("Running for this user:", user2)
# let's chat now
while True:
  user_input = input("type to ask...\n")
  # specifying some keywords as an interruption keywords
  if user_input in ("quit", "bye", "end", "thanks", "thank you"):
    print("System:: Happy to help! Bye!!!")
    break

  # adding additional config to help to get the session id
  print("AI:: ", chatModelWithPersistMemRunnable.invoke(input={"user_query": user_input}, config=config ).content)


Running for this file session Id: 6108fae5-7b40-4011-be76-130af36d813c
type to ask...
Hi I am Iron Man. I want to know what is Iron
AI::  Hi Iron Man!

type to ask...
what is Iron
AI::  Iron is a powerful and versatile metal that is used in a wide variety of applications, including:

*   **Weaponry:** It's a key component in many firearms, including the iconic Iron Man suit.
*   **Engineering:** It's used in aerospace, automotive, and other industries for its strength, durability, and resistance to corrosion.
*   **Manufacturing:** It's used in the production of various products, including clothing, electronics, and tools.
*   **Other Applications:** Iron is also used in medicine, dentistry, and other fields.
type to ask...
what is coffee?
AI::  Coffee is a beverage made from roasted coffee beans. It's a complex mixture of coffee oils, water, and sugars, and is typically served hot.
type to ask...
thanks
System:: Happy to help! Bye!!!


Now, let's run for user1 again to see if it remebers the name

In [106]:
user1= "marvel_1"
config = {"configurable":{"session_id":user1}}

print("Running for this user:", user1)
# let's chat now
while True:
  user_input = input("type to ask...\n")
  # specifying some keywords as an interruption keywords
  if user_input in ("quit", "bye", "end", "thanks", "thank you"):
    print("System:: Happy to help! Bye!!!")
    break

  # adding additional config to help to get the session id
  print("AI:: ", chatModelWithPersistMemRunnable.invoke(input={"user_query": user_input}, config=config ).content)


Running for this user: marvel_1
type to ask...
what is my name?
AI::  I am Gemma, an open-weights AI model created by the Gemma team.
type to ask...
who am I ?
AI::  I am a large language model, created by the Gemma team.
type to ask...
who I am ?
AI::  I am a large language model created by the Gemma team.
type to ask...
do you know my name ?
AI::  Yes, I know my name. I am Gemma, a large language model created by the Gemma team.
type to ask...
no my name ?
AI::  No, I do not know my name. I am a large language model.
type to ask...
thanks
System:: Happy to help! Bye!!!


In [107]:
with open(os.path.join(os.getcwd(), "user_interactions", "marvel_1.json"), "r") as jf:
  pprint(json.loads(jf.read()))

[{'data': {'additional_kwargs': {},
           'content': 'Hi My name is Captan America. I want to know the '
                      'weather today',
           'id': None,
           'name': None,
           'response_metadata': {},
           'type': 'human'},
  'type': 'human'},
 {'data': {'additional_kwargs': {},
           'content': 'Hi Captan America!\n',
           'id': 'lc_run--ca8840e9-77c0-4593-9719-9feb4b54f151-0',
           'invalid_tool_calls': [],
           'name': None,
           'response_metadata': {'created_at': '2025-11-18T12:13:53.812858984Z',
                                 'done': True,
                                 'done_reason': 'stop',
                                 'eval_count': 7,
                                 'eval_duration': 248965391,
                                 'load_duration': 1198228958,
                                 'logprobs': None,
                                 'model': 'gemma3:270m',
                                 'model_na